# Replication: "Smart Green Nudging" (von Zahn et al., 2024)
> *Marketing Science* — https://pubsonline.informs.org/doi/10.1287/mksc.2022.0393

This notebook replicates the core methodology of the paper using **simulated data**:

1. **Setup & Data Loading** — load train/test CSVs from simulation
2. **Digital Footprint Feature Engineering** — construct behavioral features
3. **ATE Estimation** — OLS and IPW estimates of nudge effect
4. **Causal Forest (CATE/HTE)** — heterogeneous treatment effects via `econml`
5. **Nudge Targeting Policy & Profit Analysis** — policy curve and profit gains

Each section ends with an **LLM interpretation call** via `litellm` + Llama3.2.

---
> **Note:** This replication uses synthetic data generated to match the paper's described data-generating process. The original retailer data is proprietary and not publicly available.

## 0. Install Dependencies

In [1]:
# Run once to install required packages
# Assumes Ollama is running locally with llama3.2 pulled: `ollama pull llama3.2`
# !pip install econml litellm pandas numpy matplotlib seaborn scikit-learn statsmodels

## 1. Setup & Data Loading

In [2]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path

# LLM
import litellm

# Causal ML
from econml.dml import CausalForestDML
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_predict

# Stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

ImportError: cannot import name '_promote' from 'scipy.spatial.transform._rotation' (c:\Users\noaht\anaconda3\envs\causal_env\Lib\site-packages\scipy\spatial\transform\_rotation.cp312-win_amd64.pyd)

In [ ]:
# ── LLM helper ────────────────────────────────────────────────────────────────
# Falls back gracefully if Ollama is not running.

def llm_interpret(section_name: str, context: str) -> str:
    """
    Call local Llama3.2 via litellm to interpret a section's results.
    Returns a plain-English paragraph suitable for a research notebook.
    """
    prompt = f"""You are a research assistant helping interpret results from a 
causal inference replication study of the paper 'Smart Green Nudging' (von Zahn et al., 2024).

Section: {section_name}

Results summary:
{context}

In 3-5 sentences, interpret these results in the context of the paper's research question 
(do personalized green nudges reduce e-commerce product returns?). 
Be concise, precise, and highlight any notable patterns or concerns."""

    try:
        response = litellm.completion(
            model="ollama_chat/llama3.2",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"[LLM unavailable — start Ollama and run `ollama pull llama3.2`]\nError: {e}"


def print_llm(section: str, context: str):
    print("\n" + "═" * 70)
    print(f"  🤖  LLM Interpretation — {section}")
    print("═" * 70)
    print(llm_interpret(section, context))
    print("═" * 70 + "\n")

In [ ]:
# ── Load simulated data ────────────────────────────────────────────────────────
# Update paths to point to your simulation output CSVs.

DATA_DIR = Path("data")          # <-- change if needed
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH  = DATA_DIR / "test.csv"

df_train = pd.read_csv(TRAIN_PATH)
df_test  = pd.read_csv(TEST_PATH)

print(f"Train shape: {df_train.shape}")
print(f"Test shape:  {df_test.shape}")
df_train.head()

In [ ]:
# ── Column configuration ───────────────────────────────────────────────────────
# EDIT these to match your simulation's actual column names.

TREATMENT_COL = 'treatment'       # Binary: 1 = received green nudge, 0 = control
OUTCOME_COL   = 'returned'        # Binary: 1 = item returned, 0 = kept

# All customer/order feature columns (digital footprint + order characteristics)
FEATURE_COLS = [c for c in df_train.columns 
                if c not in [TREATMENT_COL, OUTCOME_COL]]

print("Feature columns:")
print(FEATURE_COLS)
print(f"\nReturn rate (train): {df_train[OUTCOME_COL].mean():.3f}")
print(f"Treatment share (train): {df_train[TREATMENT_COL].mean():.3f}")

In [ ]:
# ── Descriptive overview ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Return rate by treatment arm
df_train.groupby(TREATMENT_COL)[OUTCOME_COL].mean().plot(
    kind='bar', ax=axes[0], color=['steelblue', 'seagreen'], edgecolor='white'
)
axes[0].set_title('Return Rate by Treatment Arm')
axes[0].set_xlabel('Treatment (0=Control, 1=Nudge)')
axes[0].set_ylabel('Return Rate')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[0].set_xticklabels(['Control', 'Nudge'], rotation=0)

# Sample size by treatment
df_train[TREATMENT_COL].value_counts().sort_index().plot(
    kind='bar', ax=axes[1], color=['steelblue', 'seagreen'], edgecolor='white'
)
axes[1].set_title('Sample Size by Treatment Arm')
axes[1].set_xlabel('Treatment (0=Control, 1=Nudge)')
axes[1].set_ylabel('Count')
axes[1].set_xticklabels(['Control', 'Nudge'], rotation=0)

plt.tight_layout()
plt.show()

return_control = df_train[df_train[TREATMENT_COL]==0][OUTCOME_COL].mean()
return_treated = df_train[df_train[TREATMENT_COL]==1][OUTCOME_COL].mean()
raw_diff = return_treated - return_control

print_llm(
    "Data Loading & Descriptive Overview",
    f"Train N={len(df_train)}, Test N={len(df_test)}.\n"
    f"Overall return rate: {df_train[OUTCOME_COL].mean():.3f}.\n"
    f"Control return rate: {return_control:.3f}, Nudge return rate: {return_treated:.3f}.\n"
    f"Raw difference (nudge - control): {raw_diff:.4f} ({raw_diff*100:.2f} pp).\n"
    f"Features available: {FEATURE_COLS}"
)

---
## 2. Digital Footprint Feature Engineering

The paper constructs behavioral features from customers' digital footprints — browsing sessions, past return history, basket composition, and device usage. We replicate the feature construction logic here.

In [ ]:
# ── Feature engineering ────────────────────────────────────────────────────────
# Mirrors Table 1 / Appendix features from the paper.
# Adapt these to the actual columns in your simulation CSVs.

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # --- Past return behavior ---
    if 'past_return_rate' in df.columns:
        # High returner flag (top quartile)
        q75 = df['past_return_rate'].quantile(0.75)
        df['high_returner'] = (df['past_return_rate'] >= q75).astype(int)

    # --- Bracketing behavior ---
    # Bracketing = ordering multiple sizes/colors to return most
    if 'n_items_ordered' in df.columns and 'n_items_returned' in df.columns:
        df['bracket_rate'] = df['n_items_returned'] / df['n_items_ordered'].clip(lower=1)

    # --- Order value features ---
    if 'order_value' in df.columns:
        df['log_order_value'] = np.log1p(df['order_value'])

    # --- Session engagement ---
    if 'session_duration_sec' in df.columns:
        df['log_session_duration'] = np.log1p(df['session_duration_sec'])

    # --- Mobile vs desktop ---
    if 'device_type' in df.columns:
        df['is_mobile'] = (df['device_type'].str.lower() == 'mobile').astype(int)

    return df


df_train = engineer_features(df_train)
df_test  = engineer_features(df_test)

# Refresh feature list after engineering
FEATURE_COLS = [c for c in df_train.columns 
                if c not in [TREATMENT_COL, OUTCOME_COL]]

# Drop any remaining non-numeric cols
FEATURE_COLS = [c for c in FEATURE_COLS 
                if pd.api.types.is_numeric_dtype(df_train[c])]

print(f"Features after engineering ({len(FEATURE_COLS)}):")
print(FEATURE_COLS)
df_train[FEATURE_COLS].describe().round(3)

In [ ]:
# ── Feature correlation heatmap ────────────────────────────────────────────────
top_feats = FEATURE_COLS[:min(12, len(FEATURE_COLS))]  # cap for readability

corr = df_train[top_feats + [OUTCOME_COL]].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, linewidths=0.5)
ax.set_title('Feature Correlation Matrix (incl. return outcome)')
plt.tight_layout()
plt.show()

# Top correlates with outcome
outcome_corr = corr[OUTCOME_COL].drop(OUTCOME_COL).sort_values(key=abs, ascending=False)
print("Top features correlated with return outcome:")
print(outcome_corr.head(5).round(4))

print_llm(
    "Digital Footprint Feature Engineering",
    f"Engineered {len(FEATURE_COLS)} features from digital footprint data.\n"
    f"Top correlates with return outcome:\n{outcome_corr.head(5).to_string()}"
)

---
## 3. ATE Estimation — Field Experiment

The paper estimates the **Average Treatment Effect (ATE)** of the green nudge using OLS with covariate adjustment (Table 3) and Inverse Probability Weighting (IPW) as a robustness check.

In [ ]:
# ── OLS with covariate adjustment ─────────────────────────────────────────────

X_train = df_train[FEATURE_COLS].fillna(0)
T_train = df_train[TREATMENT_COL]
Y_train = df_train[OUTCOME_COL]

X_test = df_test[FEATURE_COLS].fillna(0)
T_test = df_test[TREATMENT_COL]
Y_test = df_test[OUTCOME_COL]

# Baseline OLS (no controls)
ols_base = sm.OLS(Y_train, sm.add_constant(T_train)).fit(cov_type='HC3')

# OLS with controls
X_with_T = sm.add_constant(pd.concat([T_train, X_train], axis=1))
ols_ctrl = sm.OLS(Y_train, X_with_T).fit(cov_type='HC3')

ate_base = ols_base.params[TREATMENT_COL]
ate_ctrl = ols_ctrl.params[TREATMENT_COL]
ci_base  = ols_base.conf_int().loc[TREATMENT_COL]
ci_ctrl  = ols_ctrl.conf_int().loc[TREATMENT_COL]

print("═" * 50)
print("OLS ATE Estimates (Heteroskedasticity-Robust SE)")
print("═" * 50)
print(f"  Naive OLS (no controls): {ate_base:+.4f}  "
      f"95% CI [{ci_base[0]:+.4f}, {ci_base[1]:+.4f}]  "
      f"p={ols_base.pvalues[TREATMENT_COL]:.4f}")
print(f"  OLS + covariates:        {ate_ctrl:+.4f}  "
      f"95% CI [{ci_ctrl[0]:+.4f}, {ci_ctrl[1]:+.4f}]  "
      f"p={ols_ctrl.pvalues[TREATMENT_COL]:.4f}")
print("═" * 50)

In [ ]:
# ── IPW Estimator ─────────────────────────────────────────────────────────────
# Estimate propensity scores, then compute Horvitz-Thompson IPW ATE.

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train)

ps_model = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
ps_model.fit(X_scaled, T_train)
propensity = ps_model.predict_proba(X_scaled)[:, 1]

# Clip propensity scores for stability (common practice)
propensity = np.clip(propensity, 0.05, 0.95)

# Horvitz-Thompson IPW
ipw_treated = (T_train * Y_train / propensity).mean()
ipw_control = ((1 - T_train) * Y_train / (1 - propensity)).mean()
ate_ipw = ipw_treated - ipw_control

# Bootstrap SE for IPW
n_boot = 500
boot_ates = []
idx_all = np.arange(len(Y_train))
for _ in range(n_boot):
    idx = np.random.choice(idx_all, size=len(idx_all), replace=True)
    t_b = T_train.values[idx]; y_b = Y_train.values[idx]; p_b = propensity[idx]
    boot_ates.append(
        (t_b * y_b / p_b).mean() - ((1-t_b) * y_b / (1-p_b)).mean()
    )
ipw_se = np.std(boot_ates)
ipw_ci = (ate_ipw - 1.96*ipw_se, ate_ipw + 1.96*ipw_se)

print(f"  IPW ATE:                 {ate_ipw:+.4f}  "
      f"95% CI [{ipw_ci[0]:+.4f}, {ipw_ci[1]:+.4f}]  "
      f"Bootstrap SE={ipw_se:.4f}")
print()

# Summary plot
methods = ['OLS (naive)', 'OLS + controls', 'IPW']
ates    = [ate_base, ate_ctrl, ate_ipw]
cis     = [ci_base, ci_ctrl, ipw_ci]

fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#5B8DB8', '#4CAF82', '#E07B54']
for i, (m, a, ci) in enumerate(zip(methods, ates, cis)):
    ax.errorbar(a, i, xerr=[[a - ci[0]], [ci[1] - a]],
                fmt='o', color=colors[i], markersize=9, capsize=5, linewidth=2,
                label=m)
ax.axvline(0, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax.set_yticks(range(len(methods)))
ax.set_yticklabels(methods)
ax.set_xlabel('ATE (change in return probability)')
ax.set_title('ATE Estimates with 95% Confidence Intervals')
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=1))
plt.tight_layout()
plt.show()

print_llm(
    "ATE Estimation (OLS & IPW)",
    f"Naive OLS ATE: {ate_base:+.4f} (p={ols_base.pvalues[TREATMENT_COL]:.4f})\n"
    f"OLS + covariates ATE: {ate_ctrl:+.4f} (p={ols_ctrl.pvalues[TREATMENT_COL]:.4f})\n"
    f"IPW ATE: {ate_ipw:+.4f} (Bootstrap SE={ipw_se:.4f}, 95% CI {ipw_ci})\n"
    f"All three estimators {'agree in sign and magnitude' if all(a < 0 for a in ates) else 'show mixed results'}."
)

---
## 4. Causal Forest — Heterogeneous Treatment Effects (CATE)

The paper uses a **Causal Forest** (Wager & Athey 2018) to estimate conditional average treatment effects (CATEs) — i.e., for which customers is the nudge most effective? We replicate this using `econml.dml.CausalForestDML`.

In [ ]:
# ── Causal Forest DML ─────────────────────────────────────────────────────────
# CausalForestDML uses cross-fitting to partial out Y and T, then fits a
# causal forest on residuals — equivalent to the GRF approach in the paper.

cf_model = CausalForestDML(
    model_y = GradientBoostingRegressor(n_estimators=200, max_depth=4,
                                        random_state=RANDOM_SEED),
    model_t = GradientBoostingClassifier(n_estimators=200, max_depth=4,
                                         random_state=RANDOM_SEED),
    n_estimators    = 2000,
    min_samples_leaf= 10,
    max_depth       = None,
    max_features    = 'sqrt',
    inference       = True,
    random_state    = RANDOM_SEED,
    cv              = 5,
)

print("Fitting Causal Forest (this may take ~1-2 min)...")
cf_model.fit(Y_train.values, T_train.values, X=X_train.values)
print("Done.")

In [ ]:
# ── CATE predictions on test set ──────────────────────────────────────────────

cate_test = cf_model.effect(X_test.values)
cate_inf  = cf_model.effect_interval(X_test.values, alpha=0.05)  # 95% CI
cate_lb, cate_ub = cate_inf

df_test = df_test.copy()
df_test['cate']    = cate_test
df_test['cate_lb'] = cate_lb
df_test['cate_ub'] = cate_ub

print(f"CATE summary (test set, N={len(df_test)}):")
print(df_test['cate'].describe().round(4))

# Distribution of CATEs
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(cate_test, bins=50, color='seagreen', edgecolor='white', alpha=0.85)
axes[0].axvline(cate_test.mean(), color='red', linestyle='--',
                label=f'Mean CATE = {cate_test.mean():.4f}')
axes[0].axvline(0, color='black', linestyle='-', linewidth=0.8, alpha=0.5)
axes[0].set_xlabel('CATE (treatment effect)')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Individual CATEs (Test Set)')
axes[0].legend()
axes[0].xaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=1))

# % customers with beneficial nudge effect (CATE < 0 = fewer returns)
pct_benefit = (cate_test < 0).mean()
axes[1].pie([pct_benefit, 1-pct_benefit],
            labels=[f'Nudge reduces\nreturns ({pct_benefit:.1%})',
                    f'Nudge ineffective\nor backfires ({1-pct_benefit:.1%})'],
            colors=['seagreen', 'lightcoral'],
            startangle=90, autopct='%1.1f%%', pctdistance=0.75)
axes[1].set_title('Share of Customers Benefiting from Nudge')

plt.tight_layout()
plt.show()

In [ ]:
# ── Feature importance from causal forest ─────────────────────────────────────

feat_imp = pd.Series(
    cf_model.feature_importances_,
    index=FEATURE_COLS
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, max(4, len(FEATURE_COLS) * 0.35)))
feat_imp.head(15).plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.invert_yaxis()
ax.set_xlabel('Feature Importance')
ax.set_title('Causal Forest — Feature Importances\n(heterogeneity drivers)')
plt.tight_layout()
plt.show()

print("Top 5 heterogeneity-driving features:")
print(feat_imp.head(5).round(4))

print_llm(
    "Causal Forest / Heterogeneous Treatment Effects",
    f"Mean CATE: {cate_test.mean():+.4f} ({cate_test.mean()*100:.2f} pp).\n"
    f"CATE std: {cate_test.std():.4f} — "
    f"{'substantial' if cate_test.std() > abs(cate_test.mean()) else 'moderate'} heterogeneity.\n"
    f"{pct_benefit:.1%} of test customers have CATE < 0 (nudge reduces returns).\n"
    f"Top heterogeneity drivers: {', '.join(feat_imp.head(5).index.tolist())}"
)

---
## 5. Nudge Targeting Policy & Profit Analysis

The paper's key contribution is a **smart targeting policy**: only nudge customers where the nudge is predicted to be effective (CATE < threshold). We replicate the policy curve and profit gain analysis (Figure 4 / Table 5 in paper).

In [ ]:
# ── Cost/benefit parameters ────────────────────────────────────────────────────
# Adjust to match your simulation's economics or the paper's disclosed values.

RETURN_COST      = 15.0   # Cost to retailer per returned item (€)
NUDGE_COST       = 0.10   # Cost per nudge shown (email/banner)
AVG_ORDER_VALUE  = 80.0   # Average order value (€)

print(f"Return cost:      €{RETURN_COST:.2f}")
print(f"Nudge cost:       €{NUDGE_COST:.2f}")
print(f"Avg order value:  €{AVG_ORDER_VALUE:.2f}")

In [ ]:
# ── Policy curve: profit vs. targeting threshold ───────────────────────────────
# Sort customers by CATE (most negative = most receptive to nudge)
# and sweep the threshold from 0% to 100% targeted.

df_pol = df_test.sort_values('cate').reset_index(drop=True)
n_test = len(df_pol)

fractions   = np.linspace(0, 1, 101)
profit_smart = []
profit_univ  = []
returns_averted = []

# Baseline: no nudging
base_return_cost = df_pol[OUTCOME_COL].sum() * RETURN_COST

for frac in fractions:
    k = int(frac * n_test)
    targeted = df_pol.iloc[:k]     # top-k most nudgeable
    untargeted = df_pol.iloc[k:]

    # Expected returns averted = -CATE for targeted (if CATE < 0)
    averted = (-targeted['cate'].clip(upper=0)).sum()
    returns_averted.append(averted)

    nudge_cost_smart = k * NUDGE_COST
    profit_smart.append(averted * RETURN_COST - nudge_cost_smart)

    # Universal nudging: nudge everyone up to frac
    # (use mean ATE as uniform effect — naive policy)
    averted_univ = frac * n_test * max(0, -ate_ctrl)
    profit_univ.append(averted_univ * RETURN_COST - k * NUDGE_COST)

profit_smart = np.array(profit_smart)
profit_univ  = np.array(profit_univ)

best_smart_frac = fractions[np.argmax(profit_smart)]
best_smart_gain = profit_smart.max()
best_univ_gain  = profit_univ.max()

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(fractions * 100, profit_smart, color='seagreen',  linewidth=2.5,
        label='Smart targeting (CATE-ranked)')
ax.plot(fractions * 100, profit_univ,  color='steelblue', linewidth=2.5,
        linestyle='--', label='Universal nudging (ATE-based)')
ax.axhline(0, color='black', linestyle=':', linewidth=1, alpha=0.6)
ax.axvline(best_smart_frac * 100, color='seagreen', linestyle=':', linewidth=1.5,
           alpha=0.7, label=f'Optimal share = {best_smart_frac:.0%}')
ax.set_xlabel('% Customers Targeted with Green Nudge')
ax.set_ylabel('Profit Gain vs. No Nudging (€)')
ax.set_title('Targeting Policy Curve — Profit Gain from Smart vs. Universal Nudging')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Optimal targeting share: {best_smart_frac:.1%}")
print(f"Max profit gain (smart):    €{best_smart_gain:.2f}")
print(f"Max profit gain (universal):€{best_univ_gain:.2f}")
print(f"Value of personalization:   €{best_smart_gain - best_univ_gain:.2f}")

In [ ]:
# ── CATE by customer segment ───────────────────────────────────────────────────
# Segment customers into quartiles by CATE and show return rate by segment.

df_test['cate_quartile'] = pd.qcut(df_test['cate'], q=4,
                                    labels=['Q1\n(most\nreceptive)',
                                            'Q2', 'Q3',
                                            'Q4\n(least\nreceptive)'])

seg_stats = df_test.groupby('cate_quartile', observed=True).agg(
    n       = (OUTCOME_COL, 'count'),
    return_rate = (OUTCOME_COL, 'mean'),
    mean_cate   = ('cate', 'mean'),
).round(4)

print("CATE quartile segment statistics:")
print(seg_stats)

fig, ax = plt.subplots(figsize=(8, 4))
colors_seg = ['#2e8b57', '#5aaa78', '#f4a56a', '#e07b54']
seg_stats['mean_cate'].plot(kind='bar', ax=ax, color=colors_seg, edgecolor='white')
ax.axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.6)
ax.set_title('Mean CATE by Customer Segment (Quartile)')
ax.set_xlabel('CATE Quartile')
ax.set_ylabel('Mean CATE')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=1))
ax.set_xticklabels(seg_stats.index, rotation=0)
plt.tight_layout()
plt.show()

print_llm(
    "Nudge Targeting Policy & Profit Analysis",
    f"Optimal targeting share: {best_smart_frac:.1%} of customers.\n"
    f"Max profit gain from smart targeting: €{best_smart_gain:.2f}.\n"
    f"Max profit gain from universal nudging: €{best_univ_gain:.2f}.\n"
    f"Value of personalization over naive policy: €{best_smart_gain - best_univ_gain:.2f}.\n"
    f"CATE quartile breakdown:\n{seg_stats.to_string()}"
)

---
## 6. Summary & Conclusions

In [ ]:
# ── Final summary table ────────────────────────────────────────────────────────

summary = pd.DataFrame({
    'Metric': [
        'ATE (OLS, no controls)',
        'ATE (OLS + covariates)',
        'ATE (IPW)',
        'Mean CATE (causal forest)',
        'CATE std (heterogeneity)',
        '% customers CATE < 0',
        'Optimal targeting share',
        'Max profit gain — smart',
        'Max profit gain — universal',
        'Value of personalization',
    ],
    'Value': [
        f"{ate_base:+.4f} ({ate_base*100:+.2f} pp)",
        f"{ate_ctrl:+.4f} ({ate_ctrl*100:+.2f} pp)",
        f"{ate_ipw:+.4f} ({ate_ipw*100:+.2f} pp)",
        f"{cate_test.mean():+.4f} ({cate_test.mean()*100:+.2f} pp)",
        f"{cate_test.std():.4f}",
        f"{pct_benefit:.1%}",
        f"{best_smart_frac:.1%}",
        f"€{best_smart_gain:.2f}",
        f"€{best_univ_gain:.2f}",
        f"€{best_smart_gain - best_univ_gain:.2f}",
    ]
})

print("\n" + "═" * 60)
print("  REPLICATION SUMMARY")
print("═" * 60)
print(summary.to_string(index=False))
print("═" * 60)

In [ ]:
# ── Final LLM synthesis ────────────────────────────────────────────────────────

def llm_conclusion(summary_df: pd.DataFrame) -> str:
    prompt = f"""You are a research assistant helping write the conclusions for a 
replication study of 'Smart Green Nudging' (von Zahn et al., 2024, Marketing Science).

The paper shows that personalized green nudges — shown to customers most likely to 
respond — can reduce e-commerce product returns while being more profitable than 
universal nudging.

Here are the replication results:
{summary_df.to_string(index=False)}

Write a 5-7 sentence conclusion paragraph that:
1. States whether the replication supports the paper's main findings.
2. Highlights the most important numerical results.
3. Notes the value of causal ML personalization over naive ATE-based policies.
4. Mentions one limitation of this replication (using simulated data).
Write in formal academic prose."""

    try:
        response = litellm.completion(
            model="ollama_chat/llama3.2",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"[LLM unavailable]\nError: {e}"

print("\n" + "═" * 70)
print("  🤖  LLM-Generated Conclusion")
print("═" * 70)
print(llm_conclusion(summary))
print("═" * 70)

---
## References

- von Zahn, M., Feuerriegel, S., & Kuehl, N. (2024). **Smart Green Nudging: Reducing Product Returns Through Digital Footprints and Causal Machine Learning.** *Marketing Science*. https://doi.org/10.1287/mksc.2022.0393
- Wager, S., & Athey, S. (2018). Estimation and inference of heterogeneous treatment effects using random forests. *Journal of the American Statistical Association*, 113(523), 1228–1242.
- Athey, S., & Imbens, G. (2016). Recursive partitioning for heterogeneous causal effects. *PNAS*, 113(27), 7353–7360.
- EconML: https://econml.azurewebsites.net/
- LiteLLM: https://docs.litellm.ai/